# 1. 简介
本文是作者的第一个机器学习项目，旨在完整地探索一遍机器学习的基本步骤，包括数据的预处理，特征工程，模型训练以及模型评估。

机器学习中有一句话叫：数据决定机器学习的上限，而算法本身只是在努力逼近这个上限。因此，当训练效果较差时，除了更换模型，更重要的应该是反思自己前面的数据清洗过程。

# 2. 数据预处理

## 2.1 数据增强

In [46]:
import pandas as pd


train = pd.read_csv(r"./data/train.csv")
prior = train.groupby("Label").agg(数量=("ID", "count"))
prior

,数量
Label,
0,621
1,3873
2,570
3,6478
4,6344
5,1641
6,2881
7,1332
8,992


可以看到，不同标签之间的类别极端不平衡。在这种情况下训练模型，会有什么影响呢？我认为有两层影响：
1. 影响了每个类别先验概率，这会使得模型天然偏向多数类；
2. 影响少数类中，$P(X^{(m)} = k_m | Y = c_k)$的估计稳定性；

根据我个人的理解：
1. 对于第一个影响，如果样本收集是合理的（即训练集与真实总体分布一致），那么先验偏差不是错误，而是反映了真实情况。此时模型偏向多数类是合理的；
2. 对于第二个影响，如果各个类别的样本量足够大，那么也不构成问题；

我们假设样本中类别分布和整个样本空间的先验分布是基本一致的，于是类别不均衡不是我们的主要矛盾；而部分类的样本量非常少，如10和12，仅有100个左右，直接导致估计稳定性很差，这才是我们的主要矛盾！

一种典型的做法是对少数类进行上采样，这对于一些机器学习方法是有效的。但上采样本质这是复制自身的信息，并不会增加真正的统计显著性，仅适合在微调各样本之间的比例时使用；另外一种方法是合并少数类，或将少数类并到语义上相近的类别。不过鉴于我们这是一个比赛型项目，我们假设不能增删原始类别，因此这一方案也被否决。

经过深入搜索发现，对于数据量不大的情况，最可靠的方法是“数据增强”，因为我们的数据是文本，我们可以使用自然语言增强技术为极少数类生成新样本：
- 回译：将少数类样本翻译成另一种语言（如中文→英文→中文），生成语义相似的变体。
- 同义词替换：随机替换句子中的某些词为同义词（使用WordNet或词向量）。
- 随机插入/删除：非核心词的随机增删。

我们计划通过数据增强技术将10和12的样本量提升到300条，保证最基本的统计稳健性。注意，新增样本会改变原始数据的先验分布，这也是为什么前面要用`prior`变量将当前的数据先验情况记录下来，以便后面自行指定数据的先验分布。

另外，通常而言，数据增强技术中，回译的效果是最好的。此次数据增强以回译为主，并在样本量依然不足时使用同义词替换。

In [2]:
import os

from dotenv import load_dotenv
import openai


load_dotenv()
INTERMEDIATE_LANGUAGES = ['German', 'Spanish', 'Japanese']
TARGET_SIZE = 300

client = openai.OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

def translate_text(text: str, target_lang: str) -> str:
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "你是一个专业的翻译助手。请将用户的输入翻译成指定的语言，并只返回翻译结果，不要包含任何额外的解释或评论。"},
                {"role": "user", "content": f"Translate the following text to {target_lang}:\n\n{text}"}
            ],
            temperature=1.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return ""

def back_translate(text: str, intermediate_lang: str) -> str:
    translated_text = translate_text(text, intermediate_lang)
    back_translated_text = translate_text(translated_text, "English")
    return back_translated_text


原则上来说，我们可以让deepseek一次性翻译多条文本并返回，以节省重复的系统提示词和上下文消耗，并且速度更快。但这么做的缺点是返回格式不好控制，而且翻译质量通常得不到保证。鉴于我们的需要数据增强的数据不算太多，仅几百条，因此我们还是正常使用多轮请求完成翻译。

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

import nlpaug.augmenter.word as naw


def augment_label(df: pd.DataFrame, label: str | int, target_size: int, max_workers: int = 10):
    original_df: pd.DataFrame = df[df['Label'] == label].copy()
    original_count = len(original_df)
    if original_count >= target_size:
        return original_df
    
    need = target_size - original_count
    print(f"Label {label}: 原数量 {original_count}，需要新增 {need} 条。")
    
    all_augmented = []
    texts = original_df['content'].tolist()
    
    def process_one(text):
        results = []
        for lang in INTERMEDIATE_LANGUAGES:
            aug_text = back_translate(text, lang)
            if aug_text and aug_text != text:
                results.append(aug_text)
        print(len(all_augmented))
        return results
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_text = {executor.submit(process_one, text): text for text in texts}
        for future in as_completed(future_to_text):
            try:
                results = future.result()
                all_augmented.extend(results)
            except Exception as e:
                print(f"处理文本失败: {e}")
    
    unique_augmented = list(dict.fromkeys(all_augmented))
    print(f"Label {label}: 共生成了 {len(all_augmented)} 个变体，去重后 {len(unique_augmented)} 个。")
    if len(unique_augmented) < need:
        choice_samples = random.sample(original_df["content"].to_list(), need - len(unique_augmented))
        syn_aug = naw.SynonymAug(aug_src='wordnet')
        syn_results = syn_aug.augment(choice_samples, n=1)
        unique_augmented.extend(syn_results)
    else:
        unique_augmented = random.sample(unique_augmented, need)
    max_id = pd.to_numeric(df['ID'].str.extract(r"(\d+)")[0]).max()
    new_ids = [f"tweet_{id}" for id in range(max_id + 1, max_id + len(unique_augmented) + 1)]
    new_df = pd.DataFrame({
        'ID': new_ids,
        'content': unique_augmented,
        'Label': [label] * len(unique_augmented)
    })
    
    return new_df


augmented_10 = augment_label(train, label=10, target_size=TARGET_SIZE)
train = pd.concat([train, augmented_10], ignore_index=True)

augmented_12 = augment_label(train, label=12, target_size=TARGET_SIZE)
train = pd.concat([train, augmented_12], ignore_index=True)

train.to_csv("./data/train_augmented.csv", index=False)

## 2.2 词汇处理

In [112]:
import re
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer  # 更适合社交媒体文本
import re

# 若首次使用 nltk 资源，需下载（仅需执行一次）
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

train = pd.read_csv("./data/train_augmented.csv")

tweet_tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True)

CONTRACTION_MAP = {
    "can't": "can not",
    "won't": "will not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "mustn't": "must not",
    "mightn't": "might not",
    "needn't": "need not",
    "i'm": "i am",
    "i've": "i have",
    "i'll": "i will",
    "i'd": "i would",
    "you're": "you are",
    "you've": "you have",
    "you'll": "you will",
    "we're": "we are",
    "we've": "we have",
    "they're": "they are",
    "they've": "they have",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "here's": "here is",
    "what's": "what is",
    "who's": "who is",
    "how's": "how is",
    "let's": "let us",
}

def expand_contractions(text: str) -> str:
    for contraction, expansion in CONTRACTION_MAP.items():
        text = re.sub(r'\b' + re.escape(contraction) + r'\b', expansion, text, flags=re.IGNORECASE)
    return text

def preprocess_text(text):
    """
    修改点：
    1. 去掉停用词过滤（交给 TfidfVectorizer 的 IDF 处理）
    2. 去掉 Stemming，仅保留 Lemmatization
    3. 缩写展开而非暴力删除标点
    4. 使用 TweetTokenizer 更好处理社交媒体文本
    5. 保留 @mentions 和 #hashtags 作为特征（可选）
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # 1. 移除 URL
    text = re.sub(r'http[s]?://[^\s]+', '', text)
    
    # 2. 展开缩写（修复 "can't" → "cant" 的问题）
    text = expand_contractions(text)
    
    # 3. 使用 TweetTokenizer 分词（更适合推文风格）
    tokens = tweet_tokenizer.tokenize(text)
    
    cleaned_tokens = []
    for token in tokens:
        if token.isnumeric():
            continue
        
        if len(token) == 1 and token not in {'!', '?'}:
            continue
        
        if token.startswith('@'):
            continue
        
        if re.match(r'^[a-z!?-_]+$', token):
            stemmer = PorterStemmer()
            cleaned_tokens.append(stemmer.stem(token))
        
        elif token.startswith('#') and len(token) > 1:
            hashtag_word = token[1:]
            hashtag_word = re.sub(r'[^a-zA-Z0-9]', '', hashtag_word)
            if len(hashtag_word) > 1:
                cleaned_tokens.append(hashtag_word.lower()) 
    
    return " ".join(cleaned_tokens)


train['clean_text'] = train['content'].apply(preprocess_text)
train = train[train['clean_text'].str.strip() != '']
train


,ID,content,Label,clean_text
0,tweet_0,@tiffanylue i know i was listenin to bad habi...,0,know wa listenin to bad habit earlier and star...
1,tweet_1,Layin n bed with a headache ughhhh...waitin o...,1,layin bed with headach ughhh waitin on your call
2,tweet_2,Funeral ceremony...gloomy friday...,1,funer ceremoni gloomi friday
3,tweet_3,wants to hang out with friends SOON!,2,want to hang out with friend soon !
4,tweet_4,@dannycastillo We want to trade with someone w...,3,we want to trade with someon who ha houston ti...
...,...,...,...,...
30379,tweet_30379,"Just a moment, I'll be right back, on my iPod....",12,just moment will be right back on my ipod if i...
30380,tweet_30380,@msaysrawr *points to the question about the s...,12,point to the question about the set that just ...
30381,tweet_30381,Sitting in traffic while it rains on my car. I...,12,sit in traffic while it rain on my car wash it...
30382,tweet_30382,My stomach feels like it's about to explode fr...,12,my stomach feel like it is about to explod fro...


# 3. 特征工程

## 3.1 划分训练集和验证集

In [107]:
from sklearn.model_selection import train_test_split

X = train['clean_text']
y = train['Label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"训练集样本数: {len(X_train)}")
print(f"验证集样本数: {len(X_val)}")

训练集样本数: 24252
验证集样本数: 6064


## 3.2 词向量特征构建

In [114]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_df=0.90, 
    min_df=15,
    max_features=20000,
    sublinear_tf=True, 
)

X_train_final = vectorizer.fit_transform(X_train)
X_val_final = vectorizer.transform(X_val)

# 4. 模型训练

In [115]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import MultinomialNB


model = MultinomialNB()
model.fit(X_train_final, y_train)

y_pred = model.predict(X_val_final)
print("准确率:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

准确率: 0.31909630606860157
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       122
           1       0.35      0.12      0.18       774
           2       0.00      0.00      0.00       114
           3       0.29      0.55      0.38      1287
           4       0.31      0.57      0.40      1268
           5       0.00      0.00      0.00       328
           6       0.49      0.31      0.38       576
           7       0.00      0.00      0.00       267
           8       0.20      0.01      0.01       198
           9       0.35      0.31      0.33       781
          10       0.00      0.00      0.00        60
          11       0.00      0.00      0.00       229
          12       0.00      0.00      0.00        60

    accuracy                           0.32      6064
   macro avg       0.15      0.14      0.13      6064
weighted avg       0.27      0.32      0.27      6064



d:\Program_Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Program_Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Program_Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 5. 模型评估